# LeetCode 1404 — Number of Steps to Reduce a Number in Binary Representation to One bằng Python

Link đề: https://leetcode.com/problems/number-of-steps-to-reduce-a-number-in-binary-representation-to-one/ · Độ khó: **Medium** · Chủ đề: String, Bit Manipulation, Simulation

Skeleton LeetCode:

```python
class Solution:
    def numSteps(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu hai phép toán trên số nhị phân: **chia 2 = bỏ bit cuối**, **cộng 1 = lật dãy bit 1 cuối thành 0 và bit 0 kế tiếp thành 1** (có nhớ).
2. Đi từ mô phỏng bằng số nguyên lớn `int(s, 2)` → mô phỏng trên chuỗi → **quét một lượt với biến nhớ `carry`** `O(n)` → **công thức đóng**.
3. Biết đặt và giữ **bất biến** cho biến `carry` khi quét từ phải sang trái.
4. Tự chứng minh công thức `n + 1 + (số bit 0 giữa bit 1 đầu và bit 1 cuối)` và kiểm chứng bằng mô phỏng.
5. Tránh lỗi: `int(s)` (hệ 10), quét lẫn cả bit đầu, quên cộng `carry` cuối cùng, cộng 1 mà không lan nhớ.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s` biểu diễn một số nguyên dương. Đếm số **bước** để đưa số đó về `1` theo luật:

- Nếu số đang **chẵn** ⇒ chia cho 2.
- Nếu số đang **lẻ** ⇒ cộng thêm 1.

Đảm bảo luôn đưa được về 1.

**Ví dụ 1:** `s = "1101"` (13) → `6`: 13 lẻ → 14 → 7 lẻ → 8 → 4 → 2 → 1.

**Ví dụ 2:** `s = "10"` (2) → `1`: 2 → 1.

**Ví dụ 3:** `s = "1"` → `0`.

**Ràng buộc:** `1 <= s.length <= 500`, `s` chỉ gồm `'0'` và `'1'`, `s[0] == '1'`.

Lưu ý: số có tới 500 bit (≈ 10¹⁵⁰) — vượt xa số nguyên 64 bit ở các ngôn ngữ khác; Python thì có số nguyên lớn.

In [1]:
import random
import time
from typing import List

examples = [("1101", 6), ("10", 1), ("1", 0)]
for s, want in examples:
    print(f"s={s!r} = {int(s, 2)} -> {want}")

s='1101' = 13 -> 6
s='10' = 2 -> 1
s='1' = 1 -> 0


## 3. Kiến thức nền

### 3.1 Đổi chuỗi nhị phân ↔ số nguyên

`int(s, 2)` đọc chuỗi theo **hệ 2**; `bin(x)` trả về `'0b...'`, `format(x, 'b')` trả về chuỗi bit không tiền tố.

In [2]:
s = "1101"
x = int(s, 2)
print(x, bin(x), format(x, "b"))
print("int(s) (hệ 10, SAI ý):", int(s))

13 0b1101 1101
int(s) (hệ 10, SAI ý): 1101


### 3.2 Chẵn/lẻ nhìn bit cuối

Số nhị phân chẵn ⇔ bit cuối là `0`. Với số nguyên: `x & 1` (0 = chẵn); với chuỗi: `s[-1] == '0'`.

In [3]:
for s in ["1101", "1110", "10", "1"]:
    print(s, "chẵn" if s[-1] == "0" else "lẻ", "| x & 1 =", int(s, 2) & 1)

1101 lẻ | x & 1 = 1
1110 chẵn | x & 1 = 0
10 chẵn | x & 1 = 0
1 lẻ | x & 1 = 1


### 3.3 Chia 2 = dịch phải = bỏ bit cuối

`x >> 1` (hoặc `x // 2`) với số; `s[:-1]` với chuỗi (khi bit cuối là `0`).

In [4]:
x = 0b1110
print(format(x, "b"), "->", format(x >> 1, "b"), "| chuỗi:", "1110"[:-1])

1110 -> 111 | chuỗi: 111


### 3.4 Cộng 1 trên nhị phân: lan nhớ (carry)

Cộng 1 vào số lẻ: các bit `1` ở cuối lần lượt thành `0` (nhớ 1 sang trái), bit `0` đầu tiên gặp được thành `1`. Nếu toàn bit 1 thì thêm một bit `1` ở đầu.

- `1011 + 1 = 1100` (hai bit 1 cuối thành 0, bit 0 thành 1)
- `111 + 1 = 1000` (thêm bit mới)

In [5]:
def add_one(bits):
    """Cộng 1 vào chuỗi nhị phân, trả về chuỗi mới."""
    b = list(bits)
    i = len(b) - 1
    while i >= 0 and b[i] == "1":
        b[i] = "0"                 # 1 + 1 = 10: ghi 0, nhớ 1
        i -= 1
    if i >= 0:
        b[i] = "1"                 # 0 + nhớ 1 = 1, hết nhớ
    else:
        b.insert(0, "1")           # tràn: thêm bit mới
    return "".join(b)

for t in ["1011", "111", "1101", "1"]:
    print(t, "+ 1 =", add_one(t), "| kiểm tra:", format(int(t, 2) + 1, "b"))

1011 + 1 = 1100 | kiểm tra: 1100
111 + 1 = 1000 | kiểm tra: 1000
1101 + 1 = 1110 | kiểm tra: 1110
1 + 1 = 10 | kiểm tra: 10


### 3.5 Duyệt chuỗi ngược với `range(n - 1, 0, -1)`

`range(n − 1, 0, −1)` đi từ chỉ số cuối về **chỉ số 1** (không gồm 0). Hay dùng khi muốn xử lý mọi bit **trừ bit đầu**.

In [6]:
s = "1101"
print([(i, s[i]) for i in range(len(s) - 1, 0, -1)])

[(3, '1'), (2, '0'), (1, '1')]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** trên dạng nhị phân để thấy quy luật:

| Bước | Số | Nhị phân | Chẵn/lẻ | Thao tác |
|---|---|---|---|---|
| 0 | 13 | `1101` | lẻ | +1 |
| 1 | 14 | `1110` | chẵn | ÷2 (bỏ bit cuối) |
| 2 | 7 | `111` | lẻ | +1 (lan nhớ hết) |
| 3 | 8 | `1000` | chẵn | ÷2 |
| 4 | 4 | `100` | chẵn | ÷2 |
| 5 | 2 | `10` | chẵn | ÷2 |
| 6 | 1 | `1` | — | dừng |

Quan sát:
- Mỗi phép ÷2 **bỏ đúng một bit** cuối.
- Phép +1 chỉ xảy ra khi bit cuối là 1, và **ngay sau đó** số chắc chắn chẵn ⇒ luôn có một phép ÷2 theo sau. Vậy có thể xem "bit cuối là 1" tốn **2 bước** (+1 rồi ÷2), "bit cuối là 0" tốn **1 bước**.
- Nhưng +1 làm thay đổi các bit phía trước (lan nhớ) ⇒ cần theo dõi **nhớ** khi đi sang trái.

In [7]:
x, step = 13, 0
while x != 1:
    op = "+1" if x & 1 else "÷2"
    nx = x + 1 if x & 1 else x // 2
    step += 1
    print(f"bước {step}: {format(x, 'b'):>5s} ({x:2d}) {op} -> {format(nx, 'b'):>5s} ({nx})")
    x = nx

bước 1:  1101 (13) +1 ->  1110 (14)
bước 2:  1110 (14) ÷2 ->   111 (7)
bước 3:   111 ( 7) +1 ->  1000 (8)
bước 4:  1000 ( 8) ÷2 ->   100 (4)
bước 5:   100 ( 4) ÷2 ->    10 (2)
bước 6:    10 ( 2) ÷2 ->     1 (1)


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng số nguyên lớn

### Ý tưởng
Đổi `s` sang số nguyên `x = int(s, 2)` rồi làm đúng như đề: lặp đến khi `x == 1`, lẻ thì `+1`, chẵn thì `//2`.

In [8]:
def steps_int(s):
    x = int(s, 2)
    steps = 0
    while x != 1:
        if x & 1:
            x += 1
        else:
            x >>= 1
        steps += 1
    return steps

for s, want in examples:
    print(s, "->", steps_int(s), "| mong đợi", want)

1101 -> 6 | mong đợi 6
10 -> 1 | mong đợi 1
1 -> 0 | mong đợi 0


### Độ phức tạp và nút thắt

- Số bước ≤ `2n` (mỗi bit tốn tối đa 2 bước, xem mục 10). Mỗi phép `+1`, `>>1` trên số `n` bit tốn `O(n)` ⇒ **Time `O(n²)`**, Space `O(n)`.
- Với `n ≤ 500` vẫn chạy rất nhanh trong Python. Nhưng ở C++/Java **không có** số nguyên 500 bit ⇒ cách này **không mang đi được**. Và `O(n²)` sẽ chậm nếu `n` lên 10⁶.

Nút thắt: ta thực sự tạo ra các số trung gian khổng lồ, trong khi chỉ cần biết **bit cuối hiện tại** và **có đang nhớ hay không**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight — quét từ phải sang trái với một biến nhớ.** Thay vì thực sự cộng 1 (lan nhớ ngay), ta **trì hoãn** phần nhớ: chỉ ghi nhận `carry = 1` và cộng nó vào bit kế tiếp khi tới lượt.

Duyệt `i` từ `n − 1` về `1` (bỏ bit đầu). Giá trị thực của bit cuối hiện tại là `v = s[i] + carry`:

- `v == 0`: số chẵn ⇒ **1 bước** (÷2). `carry` giữ nguyên 0.
- `v == 2`: bit thực là `0` và vẫn nhớ 1 sang trái ⇒ số chẵn ⇒ **1 bước** (÷2). `carry` giữ nguyên 1.
- `v == 1`: số lẻ ⇒ **2 bước** (+1 rồi ÷2). Cộng 1 làm bit này thành `0` và **nhớ 1** ⇒ `carry = 1`.

Kết thúc vòng lặp còn lại bit đầu (`'1'`) cộng `carry`: nếu `carry = 1` thì số là `10` (= 2) ⇒ cần **thêm 1 bước**. Đáp án `steps + carry`.

**Quy tắc "nếu… thì…":** nếu `s[i] + carry == 1` thì `steps += 2; carry = 1`; ngược lại `steps += 1`.

**Hệ quả:** một khi `carry` thành 1 thì **không bao giờ về 0** nữa ⇒ dẫn tới công thức đóng (Cách 4).

## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng trực tiếp trên chuỗi

Không dùng số nguyên lớn: giữ `s` là **list ký tự**. Chẵn ⇒ `pop()` bit cuối. Lẻ ⇒ cộng 1 với lan nhớ (như `add_one`). Cách này "mang đi được" sang mọi ngôn ngữ, nhưng vẫn `O(n²)` trong trường hợp xấu (lan nhớ dài nhiều lần).

In [9]:
def steps_string(s):
    b = list(s)
    steps = 0
    while len(b) > 1:                      # chuỗi "1" là số 1
        if b[-1] == "0":
            b.pop()                        # ÷2
        else:
            i = len(b) - 1                 # +1 với lan nhớ
            while i >= 0 and b[i] == "1":
                b[i] = "0"
                i -= 1
            if i >= 0:
                b[i] = "1"
            else:
                b.insert(0, "1")
        steps += 1
    return steps

for s, want in examples:
    print(s, "->", steps_string(s), "| mong đợi", want)

1101 -> 6 | mong đợi 6
10 -> 1 | mong đợi 1
1 -> 0 | mong đợi 0


Time `O(n²)`: có tới ~`2n` bước, và mỗi bước có thể phải quét lại một đoạn dài (lan nhớ qua dãy bit 1, hoặc `insert(0, …)` dịch cả list). Space `O(n)`.

### Cách 3 — Quét một lượt với `carry`, `O(n)`

**Từng đoạn:**
1. `steps = 0`, `carry = 0`.
2. Với `i` từ `n − 1` về `1`: `v = int(s[i]) + carry`.
3. Nếu `v == 1`: `steps += 2`, `carry = 1`. Ngược lại: `steps += 1`.
4. Trả `steps + carry`.

In [10]:
def steps_carry(s):
    steps, carry = 0, 0
    for i in range(len(s) - 1, 0, -1):
        if int(s[i]) + carry == 1:
            steps += 2          # lẻ: +1 rồi ÷2
            carry = 1
        else:
            steps += 1          # chẵn: ÷2
    return steps + carry        # bit đầu '1' + carry: nếu thành "10" thì thêm 1 bước

for s, want in examples:
    print(s, "->", steps_carry(s), "| mong đợi", want)

1101 -> 6 | mong đợi 6
10 -> 1 | mong đợi 1
1 -> 0 | mong đợi 0


Time `O(n)`, Space `O(1)`. Bản nộp LeetCode ở mục 9.

### Cách 4 — Công thức đóng

Từ nhận xét "`carry` đã thành 1 thì giữ mãi":

- Nếu `s` là **luỹ thừa của 2** (chỉ có một bit 1): mọi bit sau là 0 ⇒ mỗi bit 1 bước ⇒ đáp án `n − 1`.
- Ngược lại, gọi `last` = vị trí bit 1 **cuối cùng** (`last ≥ 1`), `z = n − 1 − last` = số bit 0 ở cuối:
  - `z` bit 0 cuối: mỗi bit 1 bước (`carry = 0`) ⇒ `z`.
  - bit 1 tại `last`: 2 bước, `carry` thành 1.
  - các bit ở vị trí `1 … last − 1` (với `carry = 1`): bit 0 ⇒ `v = 1` ⇒ 2 bước; bit 1 ⇒ `v = 2` ⇒ 1 bước. Tổng = `(last − 1) + (số bit 0 trong đoạn đó)`.
  - cộng `carry = 1` cuối.
  - Tổng: `z + 2 + (last − 1) + zeros + 1 = n + 1 + zeros`, với `zeros` = số bit `0` trong `s[1:last]`.

In [11]:
def steps_formula(s):
    n = len(s)
    last = s.rfind("1")
    if last == 0:                      # chỉ có bit 1 đầu tiên: luỹ thừa của 2
        return n - 1
    zeros = s.count("0", 1, last)      # số bit 0 giữa bit 1 đầu và bit 1 cuối
    return n + 1 + zeros

for s, want in examples:
    print(s, "->", steps_formula(s), "| mong đợi", want)

1101 -> 6 | mong đợi 6
10 -> 1 | mong đợi 1
1 -> 0 | mong đợi 0


In [12]:
class SolutionFormula:
    def numSteps(self, s: str) -> int:
        last = s.rfind("1")
        if last == 0:
            return len(s) - 1
        return len(s) + 1 + s.count("0", 1, last)

print([SolutionFormula().numSteps(s) for s, _ in examples])

[6, 1, 0]


Time `O(n)` (`rfind`, `count` chạy trong C nên rất nhanh), Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1** `s = "1101"`:

| i | s[i] | carry trước | v = s[i]+carry | Hành động | steps | carry sau |
|---|---|---|---|---|---|---|
| 3 | 1 | 0 | 1 | lẻ: +1, ÷2 | 2 | 1 |
| 2 | 0 | 1 | 1 | lẻ: +1, ÷2 | 4 | 1 |
| 1 | 1 | 1 | 2 | chẵn: ÷2 | 5 | 1 |
| — | bit đầu `1` + carry 1 = `10` | | | ÷2 | 5 + 1 = **6** | |

In [13]:
def trace_carry(s):
    steps, carry = 0, 0
    print(f"s = {s!r}")
    print(f"{'i':>3} {'s[i]':>5} {'carry':>6} {'v':>3}  hành động        steps carry_sau")
    for i in range(len(s) - 1, 0, -1):
        c0 = carry
        v = int(s[i]) + carry
        if v == 1:
            steps += 2; carry = 1; act = "lẻ: +1 rồi ÷2"
        else:
            steps += 1; act = "chẵn: ÷2      "
        print(f"{i:>3} {s[i]:>5} {c0:>6} {v:>3}  {act}   {steps:>5} {carry:>5}")
    print(f"bit đầu '1' + carry {carry} -> {'thêm 1 bước' if carry else 'đã là 1'}; đáp án = {steps + carry}")
    return steps + carry

trace_carry("1101")
print()
trace_carry("101100")

s = '1101'
  i  s[i]  carry   v  hành động        steps carry_sau
  3     1      0   1  lẻ: +1 rồi ÷2       2     1
  2     0      1   1  lẻ: +1 rồi ÷2       4     1
  1     1      1   2  chẵn: ÷2             5     1
bit đầu '1' + carry 1 -> thêm 1 bước; đáp án = 6

s = '101100'
  i  s[i]  carry   v  hành động        steps carry_sau
  5     0      0   0  chẵn: ÷2             1     0
  4     0      0   0  chẵn: ÷2             2     0
  3     1      0   1  lẻ: +1 rồi ÷2       4     1
  2     1      1   2  chẵn: ÷2             5     1
  1     0      1   1  lẻ: +1 rồi ÷2       7     1
bit đầu '1' + carry 1 -> thêm 1 bước; đáp án = 8


8

So sánh với mô phỏng thật trên `"101100"` (44) để thấy từng dòng của bảng ứng với bước nào:

In [14]:
def trace_simulation(s):
    x, step = int(s, 2), 0
    while x != 1:
        op = "+1" if x & 1 else "÷2"
        x = x + 1 if x & 1 else x >> 1
        step += 1
        print(f"bước {step:2d}: {op} -> {format(x, 'b'):>8s}")
    return step

print("tổng:", trace_simulation("101100"), "| công thức:", steps_formula("101100"))

bước  1: ÷2 ->    10110
bước  2: ÷2 ->     1011
bước  3: +1 ->     1100
bước  4: ÷2 ->      110
bước  5: ÷2 ->       11
bước  6: +1 ->      100
bước  7: ÷2 ->       10
bước  8: ÷2 ->        1
tổng: 8 | công thức: 8


Công thức cho `"101100"`: `n = 6`, bit 1 cuối ở vị trí 3, `s[1:3] = "01"` có 1 bit 0 ⇒ `6 + 1 + 1 = 8` ✔.

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def numSteps(self, s: str) -> int:
        steps, carry = 0, 0
        for i in range(len(s) - 1, 0, -1):      # mọi bit trừ bit đầu, từ phải sang trái
            if int(s[i]) + carry == 1:          # bit thực là 1 -> số lẻ
                steps += 2                      # +1 rồi ÷2
                carry = 1
            else:                               # bit thực là 0 -> số chẵn
                steps += 1                      # ÷2
        return steps + carry                    # bit đầu: "1" (xong) hoặc "10" (thêm 1 bước)

for s, want in examples:
    assert Solution().numSteps(s) == want
print("Solution đúng với 3 ví dụ của đề")

Solution đúng với 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

Gọi `x` là số hiện tại. Ta chứng minh **bất biến**: trước khi xử lý vị trí `i` (đi từ phải sang trái), số hiện tại bằng `x_i = (giá trị của s[0..i]) + carry`, và `steps` đúng bằng số bước đã thực hiện để đi từ số ban đầu tới `x_i`.

- **Khởi đầu** (`i = n − 1`): `x = int(s)`, `carry = 0`, `steps = 0` ✔.
- **Bước giữ bất biến**: bit cuối thực sự của `x_i` là `(s[i] + carry) mod 2`.
  - Nếu `s[i] + carry = 0`: `x_i` chẵn ⇒ một phép ÷2 ⇒ còn `s[0..i−1]` với `carry = 0` ✔.
  - Nếu `s[i] + carry = 2`: bit cuối là 0 và có nhớ 1 sang vị trí `i − 1`, tức `x_i = s[0..i−1]·2 + 2·carry_in`… viết gọn: `x_i = 2·(s[0..i−1] + 1)` ⇒ chẵn ⇒ ÷2 ⇒ `s[0..i−1] + 1` = trạng thái mới với `carry = 1` ✔.
  - Nếu `s[i] + carry = 1`: `x_i` lẻ ⇒ +1 làm bit cuối thành 0 và nhớ 1 ⇒ `x_i + 1 = 2·(s[0..i−1] + 1)` ⇒ ÷2 ⇒ `s[0..i−1] + 1`: 2 bước, `carry = 1` ✔.
- **Kết thúc** (`i = 0`): số hiện tại là `1 + carry`. Nếu `carry = 0` ⇒ đã là 1. Nếu `carry = 1` ⇒ số là 2 ⇒ thêm đúng 1 phép ÷2 ✔.

Mỗi bước trong lập luận đúng là bước bắt buộc của đề (không có lựa chọn), nên số bước đếm được là **duy nhất** và chính xác. Đồng thời ta thấy `số bước ≤ 2(n − 1) + 1`.

**Công thức (Cách 4)** suy trực tiếp từ bất biến trên, vì `carry` chỉ chuyển 0 → 1 tại bit 1 cuối cùng và không bao giờ trở về 0 (khi `carry = 1`, `v ∈ {1, 2}` ⇒ `carry` sau vẫn 1).

## 11. Phân tích Time / Space

Gọi `n = len(s)` (≤ 500).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. `int(s, 2)` mô phỏng | `O(n²)` (≤ 2n bước × `O(n)` mỗi phép số lớn) | `O(n)` | cần số nguyên lớn |
| 2. Mô phỏng trên chuỗi | `O(n²)` | `O(n)` | mang được sang mọi ngôn ngữ |
| 3. Quét với `carry` | `O(n)` | `O(1)` | chuẩn |
| 4. Công thức đóng | `O(n)` | `O(1)` | ngắn nhất, cần chứng minh |

Số bước tối đa: chuỗi `"1"+"0"*(n−2)+"1"` cho `n + 1 + (n − 2) = 2n − 1` bước.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [16]:
ALL = {
    "int": steps_int,
    "string": steps_string,
    "carry": steps_carry,
    "formula": steps_formula,
    "formula_submit": lambda s: SolutionFormula().numSteps(s),
    "submit": lambda s: Solution().numSteps(s),
}
edge = [
    ("1", 0), ("10", 1), ("11", 3),        # 3 -> 4 -> 2 -> 1
    ("100", 2), ("111", 4),                # 7 -> 8 -> 4 -> 2 -> 1
    ("1111", 5), ("1000", 3),
    ("101", 5),                            # 5 -> 6 -> 3 -> 4 -> 2 -> 1
    ("1" + "0" * 499, 499),                # luỹ thừa 2 lớn nhất
    ("1" * 500, 501),                      # toàn bit 1
    ("1" + "0" * 498 + "1", 999),          # nhiều bước nhất: 2n - 1
]
for s, want in edge:
    assert steps_int(s) == want, (s[:10], steps_int(s))
for name, f in ALL.items():
    for s, want in examples + edge:
        assert f(s) == want, (name, s[:20])
print("Mọi cách đúng với", len(examples + edge), "test (nhãn edge kiểm chứng bằng mô phỏng int)")

Mọi cách đúng với 14 test (nhãn edge kiểm chứng bằng mô phỏng int)


### 12.2 Vét cạn mọi chuỗi ngắn + ngẫu nhiên dài (seed cố định)

In [17]:
cnt = 0
for x in range(1, 1 << 14):                # mọi số từ 1 đến 2^14 - 1
    s = format(x, "b")
    want = steps_int(s)
    for name, f in ALL.items():
        assert f(s) == want, (name, s)
    cnt += 1
rng = random.Random(1404)
for _ in range(1000):
    n = rng.randint(1, 500)
    s = "1" + "".join(rng.choice("01") for _ in range(n - 1))
    want = steps_int(s)
    for name, f in ALL.items():
        assert f(s) == want, (name, s[:20])
print(f"vét cạn {cnt} chuỗi ngắn + 1000 chuỗi ngẫu nhiên dài: mọi cách khớp")

vét cạn 16383 chuỗi ngắn + 1000 chuỗi ngẫu nhiên dài: mọi cách khớp


### 12.3 Hiệu năng (thử cả `n = 10^5`, ngoài ràng buộc, để thấy khác biệt `O(n²)` và `O(n)`)

In [18]:
rng = random.Random(9)
s_big = "1" + "".join(rng.choice("01") for _ in range(10**5 - 1))
for name in ["int", "carry", "formula"]:
    t = time.perf_counter()
    r = ALL[name](s_big)
    print(f"{name:8s}: {r} ({time.perf_counter() - t:.3f}s)")

int     : 150182 (0.178s)
carry   : 150182 (0.008s)
formula : 150182 (0.000s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `int(s)` thay vì `int(s, 2)`

Đọc chuỗi như số **hệ 10**: `"1101"` thành một nghìn một trăm lẻ một.

In [19]:
def wrong_decimal(s):
    x = int(s)                                # SAI: hệ 10
    steps = 0
    while x != 1:
        x = x + 1 if x & 1 else x >> 1
        steps += 1
    return steps

for s, want in examples:
    print(s, "| sai:", wrong_decimal(s), "| đúng:", want)

1101 | sai: 18 | đúng: 6
10 | sai: 6 | đúng: 1
1 | sai: 0 | đúng: 0


**Sửa:** `int(s, 2)`.

### Lỗi 2 — Quét cả bit đầu (chỉ số 0)

Vòng `range(len(s) − 1, −1, −1)` xử lý luôn bit đầu như các bit khác ⇒ đếm thừa bước (số 1 cũng bị "+1 rồi ÷2").

In [20]:
def wrong_include_msb(s):
    steps, carry = 0, 0
    for i in range(len(s) - 1, -1, -1):       # SAI: gồm cả i = 0
        if int(s[i]) + carry == 1:
            steps += 2; carry = 1
        else:
            steps += 1
    return steps + carry

for s, want in examples:
    print(s, "| sai:", wrong_include_msb(s), "| đúng:", want)

1101 | sai: 7 | đúng: 6
10 | sai: 4 | đúng: 1
1 | sai: 3 | đúng: 0


**Sửa:** dừng ở `i = 1`; bit đầu xử lý riêng bằng `+ carry`.

### Lỗi 3 — Quên cộng `carry` cuối cùng

In [21]:
def wrong_forget_carry(s):
    steps, carry = 0, 0
    for i in range(len(s) - 1, 0, -1):
        if int(s[i]) + carry == 1:
            steps += 2; carry = 1
        else:
            steps += 1
    return steps                              # SAI: thiếu + carry

for s, want in examples + [("11", 3), ("111", 4)]:
    print(s, "| sai:", wrong_forget_carry(s), "| đúng:", want)

1101 | sai: 5 | đúng: 6
10 | sai: 1 | đúng: 1
1 | sai: 0 | đúng: 0
11 | sai: 2 | đúng: 3
111 | sai: 3 | đúng: 4


Với `"1101"` vòng lặp kết thúc ở trạng thái "10" (bit đầu 1 + nhớ 1) mà không đếm phép ÷2 cuối. **Sửa:** `return steps + carry`.

### Lỗi 4 — Cộng 1 trên chuỗi mà không lan nhớ

Viết "số lẻ thì đổi bit cuối thành 0 rồi đổi bit kế thành 1" — chỉ đúng khi bit kế là 0.

In [22]:
def wrong_no_propagation(s, limit=10000):
    b = list(s)
    steps = 0
    while len(b) > 1 and steps < limit:
        if b[-1] == "0":
            b.pop()
        else:
            b[-1] = "0"
            if len(b) >= 2:
                b[-2] = "1"                   # SAI: không lan nhớ khi b[-2] đã là 1
            else:
                b.insert(0, "1")
        steps += 1
    return steps if len(b) == 1 else f"không về 1 sau {limit} bước"

for s in ["1101", "11", "101", "1111", "1010"]:
    print(s, "| sai:", wrong_no_propagation(s), "| đúng:", Solution().numSteps(s))

1101 | sai: 6 | đúng: 6
11 | sai: 2 | đúng: 3
101 | sai: 4 | đúng: 5
1111 | sai: 6 | đúng: 5
1010 | sai: 5 | đúng: 6


`"11" + 1` phải là `"100"` (= 4), bản sai cho `"10"` (= 2) vì `b[-2]` vốn đã là 1 nên phải lan nhớ tiếp ⇒ đếm **thiếu** bước; còn `"1111" + 1` phải là `"10000"`, bản sai cho `"1110"` ⇒ đếm **thừa** bước. (`"1101"` tình cờ vẫn đúng — một ví dụ chạy đúng chưa chứng minh được gì.) **Sửa:** vòng `while` lan nhớ qua mọi bit 1 ở cuối (hàm `add_one`).

### Lỗi 5 — Gán sai `carry` khi `v == 2`

Nghĩ rằng "đã chia 2 thì hết nhớ" và đặt `carry = 0` khi `v == 2`. Thực ra `1 + 1 = 10`: bit hiện tại 0 nhưng **nhớ 1 vẫn còn** sang trái.

In [23]:
def wrong_reset_carry(s):
    steps, carry = 0, 0
    for i in range(len(s) - 1, 0, -1):
        v = int(s[i]) + carry
        if v == 1:
            steps += 2; carry = 1
        else:
            steps += 1
            carry = 0                         # SAI khi v == 2
    return steps + carry

for s, want in examples + [("111", 4), ("1011", 5)]:
    print(s, "| sai:", wrong_reset_carry(s), "| đúng:", Solution().numSteps(s))

1101 | sai: 5 | đúng: 6
10 | sai: 1 | đúng: 1
1 | sai: 0 | đúng: 0
111 | sai: 3 | đúng: 4
1011 | sai: 4 | đúng: 6


**Sửa:** khi `v == 2`, giữ nguyên `carry = 1` (hoặc viết tổng quát `carry = v // 2` sau khi xử lý).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mô phỏng phép toán trên số nhị phân dạng chuỗi"** ⇒ quét từ phải sang trái với biến `carry`.
- Chia 2 = bỏ bit cuối; nhân 2 = thêm `0`; cộng 1 = lan nhớ qua dãy 1 cuối.
- Khi một thao tác **luôn kéo theo** thao tác khác (+1 luôn kèm ÷2), gộp chúng thành một "bước kép" để đếm.
- Nếu thấy một biến trạng thái chỉ đổi **một chiều** (0 → 1 rồi giữ), thường có **công thức đóng**.
- Python có số nguyên lớn ⇒ mô phỏng `int(s, 2)` là cách kiểm thử nhanh và chắc chắn đúng.

## 15. Biến thể của bài

1. **Số lẻ thì trừ 1** (thay vì cộng 1): mỗi bit 1 (trừ bit đầu) tốn 2 bước, bit 0 tốn 1 ⇒ `(n − 1) + (số bit 1 − 1)` (LeetCode 1342 *Number of Steps to Reduce a Number to Zero* là bài họ hàng với số nguyên thường).
2. **Được chọn +1 hoặc −1 khi lẻ, tối thiểu số bước** (LeetCode 397 *Integer Replacement*) ⇒ tham lam theo hai bit cuối (Bài 5 mục 17).
3. **Cộng hai chuỗi nhị phân** (LeetCode 67 *Add Binary*) ⇒ cùng kỹ thuật `carry`.
4. **Đếm bước của dãy Collatz** (lẻ ⇒ `3x + 1`) — không còn công thức đóng, phải mô phỏng.
5. **In ra dãy trạng thái nhị phân** qua từng bước (Bài 4).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `int(s, 2)` | mô phỏng trên số nguyên lớn | `O(n²)` | `O(n)` | Python, kiểm thử nhanh |
| 2. Mô phỏng chuỗi | `pop` khi chẵn, lan nhớ khi lẻ | `O(n²)` | `O(n)` | ngôn ngữ không có số lớn, cần xem trạng thái |
| 3. Quét `carry` | mỗi bit 1 hoặc 2 bước | `O(n)` | `O(1)` | **mặc định**, phỏng vấn |
| 4. Công thức | `n + 1 + zeros` (hoặc `n − 1`) | `O(n)` | `O(1)` | khi đã chứng minh được, code ngắn nhất |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().numSteps("1111")` và `Solution().numSteps("100000")` bằng bao nhiêu?

### Bài 2 (tính tay)
Lập bảng `i, s[i], carry, v, steps` của Cách 3 cho `s = "10110"` và so với công thức.

### Bài 3 (điền TODO)
Hoàn thiện `num_steps_todo(s)` theo Cách 3.

### Bài 4 (tự viết lại)
Viết `states(s)` trả về **danh sách các chuỗi nhị phân** qua từng bước (gồm cả `s` ban đầu và `"1"` cuối), chỉ thao tác trên chuỗi, không dùng `int(s, 2)`.

### Bài 5 (mini challenge)
Biến thể *Integer Replacement* trên chuỗi nhị phân: khi số lẻ được **chọn** `+1` hoặc `−1`. Viết `min_steps(s)` trả về số bước **ít nhất** để về 1 (chẵn vẫn bắt buộc ÷2).

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def num_steps_todo(s):
    steps, carry = 0, 0
    for i in range(len(s) - 1, 0, -1):
        # TODO: v = bit hiện tại + carry
        # TODO: nếu v == 1: steps += 2, carry = 1; ngược lại steps += 1
        pass
    # TODO: trả về steps + carry
    pass

# Bài 4 — TODO
def states(s):
    # TODO: lặp đến khi chuỗi là "1": chẵn -> bỏ bit cuối; lẻ -> add_one
    pass

# Bài 5 — TODO
def min_steps(s):
    # TODO: x lẻ: x == 3 -> trừ 1; hai bit cuối "11" -> cộng 1; ngược lại trừ 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `"1111"` có `last = 3`, không có bit 0 ở giữa. (2) Công thức `4 + 1 + 0 = 5` (15 → 16 → 8 → 4 → 2 → 1). (3) `"100000"` là luỹ thừa 2 ⇒ `6 − 1 = 5`.

**Bài 2.** (1) `i=4`: bit 0, carry 0 ⇒ v=0, steps=1. (2) `i=3`: bit 1 ⇒ v=1, steps=3, carry=1; `i=2`: bit 1 ⇒ v=2, steps=4. (3) `i=1`: bit 0 ⇒ v=1, steps=6, carry=1 ⇒ đáp án 7 = `5 + 1 + 1` (một bit 0 trong `s[1:3] = "01"`).

**Bài 3.** (1) `v = int(s[i]) + carry`. (2) Hai nhánh như mô tả. (3) `return steps + carry`.

**Bài 4.** (1) `cur = s; out = [cur]`. (2) `cur = cur[:-1] if cur[-1] == "0" else add_one(cur)`. (3) Kiểm tra `len(out) − 1 == numSteps(s)`.

**Bài 5.** (1) Thử đệ quy có nhớ trên số nguyên để có đáp án chuẩn. (2) Quy tắc tham lam: lẻ và hai bit cuối là `11` (trừ trường hợp số 3) ⇒ `+1` tạo nhiều bit 0 hơn; ngược lại `−1`. (3) So tham lam với đệ quy cho mọi `x ≤ 3000`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1, 2
assert Solution().numSteps("1111") == 5
assert Solution().numSteps("100000") == 5
assert Solution().numSteps("10110") == 7 == steps_formula("10110") == steps_int("10110")
print("Bài 1, 2: đúng")

Bài 1, 2: đúng


In [26]:
# Bài 3 — đáp án
def num_steps_sol(s):
    steps, carry = 0, 0
    for i in range(len(s) - 1, 0, -1):
        v = int(s[i]) + carry
        if v == 1:
            steps += 2
            carry = 1
        else:
            steps += 1
    return steps + carry

for x in range(1, 5000):
    s = format(x, "b")
    assert num_steps_sol(s) == steps_int(s)
print("Bài 3: đúng")

Bài 3: đúng


In [27]:
# Bài 4 — đáp án
def states_sol(s):
    cur, out = s, [s]
    while cur != "1":
        cur = cur[:-1] if cur[-1] == "0" else add_one(cur)
        out.append(cur)
    return out

print(states_sol("1101"))
assert [int(t, 2) for t in states_sol("1101")] == [13, 14, 7, 8, 4, 2, 1]
rng = random.Random(4)
for _ in range(300):
    s = "1" + "".join(rng.choice("01") for _ in range(rng.randint(0, 40)))
    st = states_sol(s)
    assert st[0] == s and st[-1] == "1" and len(st) - 1 == Solution().numSteps(s)
print("Bài 4: đúng")

['1101', '1110', '111', '1000', '100', '10', '1']
Bài 4: đúng


In [28]:
# Bài 5 — đáp án
from functools import lru_cache

@lru_cache(maxsize=None)
def min_steps_ref(x):
    if x == 1:
        return 0
    if x % 2 == 0:
        return 1 + min_steps_ref(x // 2)
    return 1 + min(min_steps_ref(x - 1), min_steps_ref(x + 1))

def min_steps_sol(s):
    x, steps = int(s, 2), 0
    while x != 1:
        if x % 2 == 0:
            x //= 2
        elif x == 3 or x % 4 == 1:          # hai bit cuối "01" (hoặc số 3) -> trừ 1
            x -= 1
        else:                               # hai bit cuối "11" -> cộng 1
            x += 1
        steps += 1
    return steps

for x in range(1, 3000):
    assert min_steps_sol(format(x, "b")) == min_steps_ref(x), x
print("ví dụ: '1101' (13) ->", min_steps_sol("1101"), "bước tối thiểu (bài gốc bắt buộc +1 cần", Solution().numSteps("1101"), ")")
print("Bài 5: đúng")

ví dụ: '1101' (13) -> 5 bước tối thiểu (bài gốc bắt buộc +1 cần 6 )
Bài 5: đúng


Lưu ý `min_steps_ref` chỉ dùng cho số nhỏ: đệ quy `x + 1` có thể đi lên nhưng chỉ tăng một bậc, nên số trạng thái vẫn hữu hạn và nhỏ.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template quét nhị phân với nhớ:**

```python
steps, carry = 0, 0
for i in range(len(s) - 1, 0, -1):      # bỏ bit đầu
    if int(s[i]) + carry == 1:          # lẻ
        steps += 2; carry = 1           # +1 rồi ÷2
    else:                               # chẵn (v = 0 hoặc 2)
        steps += 1                      # ÷2, carry giữ nguyên
return steps + carry
```

**Công thức:** luỹ thừa 2 ⇒ `n − 1`; ngược lại `n + 1 + (số '0' trong s[1:last])`.

**Checklist trước khi submit:**
- [ ] Đọc nhị phân bằng `int(s, 2)` nếu mô phỏng (không phải `int(s)`).
- [ ] Vòng lặp dừng ở `i = 1`, không xử lý bit đầu.
- [ ] Khi `v == 2` **không** đặt lại `carry = 0`.
- [ ] Trả `steps + carry`.
- [ ] Đã thử `"1" → 0`, `"10" → 1`, `"11" → 3`, `"1111" → 5`.
- [ ] Đúng chữ ký: `numSteps(self, s: str) -> int`.